# Somus — Fine-Tune LFM2.5-1.2B-Instruct with Unsloth

This notebook fine-tunes **LFM2.5-1.2B-Instruct** following the [official Unsloth LFM2.5 guide](https://unsloth.ai/docs/models/tutorials/lfm2.5).

### Prerequisites
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `somus_lfm_finetune_dataset.jsonl` to the Colab file browser (left sidebar)

---
## Step 1: Install Dependencies

In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

if major_version >= 8:
    !pip install --no-deps packaging ninja einops "flash-attn>=2.6.3"

!pip install --no-deps "trl>=0.8.6" peft accelerate bitsandbytes datasets

---
## Step 2: Load Base Model

**CRITICAL**: LFM requires `load_in_4bit=False` — its hybrid architecture breaks with 4-bit quantization during training.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096   # Official recommendation
dtype = None             # Auto-detect
load_in_4bit = False     # MUST be False for LFM2.5!

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="LiquidAI/LFM2.5-1.2B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

---
## Step 3: Attach LoRA Adapters

**CRITICAL**: LFM uses different layer names than standard transformers.
We must target `out_proj`, `in_proj`, `w1`, `w2`, `w3` — NOT the standard `o_proj`, `gate_proj`, `up_proj`, `down_proj`.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj",  # Attention queries/keys/values
        "out_proj", "in_proj",          # LFM-specific attention I/O
        "w1", "w2", "w3",              # LFM-specific MLP layers
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA attached: training {trainable/1e6:.1f}M / {total/1e6:.1f}M params ({100*trainable/total:.1f}%)")

---
## Step 4: Load & Prepare Dataset

**Upload** `somus_lfm_finetune_dataset.jsonl` to the Colab file browser before running this cell.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = load_dataset(
    "json",
    data_files="somus_lfm_finetune_dataset.jsonl",
    split="train",
)

dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"Dataset loaded: {len(dataset)} examples")
print(f"\n--- Sample ---")
print(dataset[0]["text"][:500])

---
## Step 5: Train!

This will take approximately **20-30 minutes** on a T4 GPU.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"Total time: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

---
## Step 6: Test the Fine-Tuned Model

In [ ]:
import json

FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = '''Extract transaction details from this bank SMS as JSON.

IMPORTANT RULES:
- isFinancial=true ONLY if money HAS ALREADY been debited/credited (past tense)
- isFinancial=false for: reminders, upcoming payments, "will be debited", "due on", "scheduled", renewals, OTPs, promotions
- amount: the EXACT number from SMS. Never invent or guess amounts.
- merchant: the actual payee/store/recipient name, NOT the SMS sender code
- Output ONLY the JSON, nothing else.'''

test_cases = [
    'SMS sender: JX-HDFCBK body: "HDFC Bank: Rs 450.00 debited from a/c XX4321 on 07-Apr-25 to SWIGGY (UPI Ref No 412345678901)."',
    'SMS sender: AD-ICICIB body: "Your EMI of Rs.5,432 will be debited from A/c XX1234 on 15-Apr. Pls maintain sufficient balance."',
    'SMS sender: AD-HDFCBK body: "387707 is your OTP for HDFC Bank NetBanking login. Valid for 2 mins. NEVER share OTP."',
    'SMS sender: AX-HDFCBK body: "Update! INR 56,370.00 deposited in HDFC Bank A/c XX1234 on 02-MAR-25 for NEFT Cr-YESB-AGRIM WHOLESALE-DEVESH YADAV."',
    'SMS sender: AD-ZEPTON body: "Rs.50 free cash is waiting in your wallet. Use it at Zepto now."',
    'SMS sender: TX-KOTAKB body: "Auto-Pay of INR 149.00 to Youtube debited via Kotak Card XX5678 on 29/03/2025."',
]

print("=" * 80)
print("MODEL TEST RESULTS")
print("=" * 80)

for i, sms in enumerate(test_cases, 1):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": sms},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.1,
        top_p=0.1,
        repetition_penalty=1.05,
        use_cache=True,
    )

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    short_body = sms.split('body: ')[1][:80]

    print(f"\n--- Test {i} ---")
    print(f"SMS: {short_body}")
    print(f"Output: {response.strip()}")

    try:
        parsed = json.loads(response.strip())
        is_fin = parsed.get('isFinancial', '?')
        verdict = 'TRANSACTION' if is_fin else 'REJECTED'
        print(f"Verdict: {verdict}")
    except json.JSONDecodeError:
        print(f"Could not parse JSON")

---
## Step 7: Export to GGUF

In [ ]:
# Save LoRA adapters
model.save_pretrained("somus_lfm_lora")
tokenizer.save_pretrained("somus_lfm_lora")

# Merge and export to GGUF Q4_K_M (optimal for mobile)
model.save_pretrained_gguf(
    "somus-lfm-1.2b-sms",
    tokenizer,
    quantization_method="q4_k_m",
)

print("\nGGUF file saved to: somus-lfm-1.2b-sms/")
print("Download the .gguf file from the folder in the left sidebar.")
print("Then update LeapService.kt to point to this new model file.")

---
## Step 8 (Optional): Push to Hugging Face Hub

In [ ]:
# Uncomment and fill in your details to push to HuggingFace
# model.push_to_hub_gguf(
#     "YOUR_HF_USERNAME/somus-lfm-1.2b-sms",
#     tokenizer,
#     quantization_method="q4_k_m",
#     token="YOUR_HF_TOKEN",
#     private=True,
# )
# print("Pushed to HuggingFace!")